# Февраль 2026: атрибуты дашборда (минимум комментариев)


In [ ]:
import re
from decimal import Decimal, InvalidOperation
import pandas as pd
from rail_connectors.connection import connect

pd.options.display.max_columns = None
pd.options.display.width = None


In [ ]:
month_start = '2026-02-01'
month_end = (pd.to_datetime(month_start) + pd.offsets.MonthEnd(0)).strftime('%Y-%m-%d')
aur_rate = 1926.0

imp = connect(
    to='IMPALA',
    extra_options={'db': 'sandbox_ai'},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={'user_name': 'Shestopalov-VYur'}
)
imp._init_connection()
with imp:
    imp.execute('invalidate metadata ods_alpha.scd1_agreements')
    imp.execute('invalidate metadata ods_alpha.scd1_companies')
    imp.execute('invalidate metadata ods_alpha.scd1_merchants')
    imp.execute('invalidate metadata ods_alpha.scd1_pos_terminals')
    imp.execute('invalidate metadata ods_alpha.scd1_trx')
    imp.execute('invalidate metadata ods_alpha.scd1_trx_acq')
    imp.execute('invalidate metadata ods_alpha.scd1_trx_int')
    imp.execute('invalidate metadata ods.scd1_z_r2_ip_merchants')
    imp.execute('invalidate metadata ods.scd1_z_r2_tariff_tune')
    imp.execute('invalidate metadata ods.scd1_z_r2_tariff_fix')
    imp.execute('invalidate metadata ods.scd1_z_r2_tariff_plan')
    imp.execute('invalidate metadata ods.scd1_z_depart')
    imp.execute('invalidate metadata ods.scd1_z_branch')
    imp.execute('invalidate metadata ods.scd1_z_cl_corp')
    imp.execute('invalidate metadata sandbox_ai.shestopalov_terminal_amortization_oneoff')

def normalize_numeric_str(v):
    if pd.isna(v):
        return None
    s = str(v).strip().replace(' ', '').replace('\xa0', '')
    if s == '' or s.lower() in {'nan', 'none', 'null'}:
        return None
    s = s.replace(',', '.')
    if re.search(r"[eE][+-]?\d+$", s):
        try:
            s = format(Decimal(s), 'f')
        except InvalidOperation:
            return None
    s = re.sub(r"\.0+$", '', s)
    s = re.sub(r"\D", '', s)
    return s if s else None

def normalize_inn(v):
    s = normalize_numeric_str(v)
    if s is None:
        return None
    if len(s) in (10, 12):
        return s
    if len(s) == 9:
        return s.zfill(10)
    if len(s) == 11:
        return s.zfill(12)
    return None

def normalize_contract(v):
    if pd.isna(v):
        return None
    s = str(v).strip().replace('\xa0', ' ')
    s = re.sub(r"\s+", ' ', s)
    if s == '' or s.lower() in {'nan', 'none', 'null'}:
        return None
    return s.lower()

def normalize_agr_id(v):
    return normalize_numeric_str(v)


In [ ]:
sql_attrs = f"""
with base as (
    select
        case
          when length(regexp_replace(trim(c.c_inn), '[^0-9]', '')) = 9 then lpad(regexp_replace(trim(c.c_inn), '[^0-9]', ''), 10, '0')
          when length(regexp_replace(trim(c.c_inn), '[^0-9]', '')) = 11 then lpad(regexp_replace(trim(c.c_inn), '[^0-9]', ''), 12, '0')
          else regexp_replace(trim(c.c_inn), '[^0-9]', '')
        end as inn_norm,
        cast(c.c_cmp_name as string) as client_name,
        lower(trim(cast(a.c_agr_number as string))) as contract_norm,
        cast(a.c_agr_number as string) as contract_number,
        cast(a.abs_agr_id as string) as agr_id_norm,
        cast(a.n_agr as string) as n_agr,
        cast(a.d_valid_from as date) as d_valid_from,
        cast(a.d_valid_to as date) as d_valid_to,
        a.n_cmp_client
    from ods_alpha.scd1_agreements a
    join ods_alpha.scd1_companies c
      on c.n_cmp = a.n_cmp_client
    where a.acq_class = 'SA'
      and cast(a.d_valid_from as date) <= cast('{month_start}' as date)
      and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{month_start}' as date))
      and coalesce(a.ods_deleted_flg, '0') <> '1'
      and coalesce(c.ods_deleted_flg, '0') <> '1'
),
r2_one as (
    select cast(m.id as string) as agr_id_norm, m.c_cl_org, m.c_depart, m.c_tariff_plan
    from ods.scd1_z_r2_ip_merchants m
),
dep_one as (
    select d.id, d.c_name as vsp_name, d.c_num as vsp_code, d.c_filial
    from ods.scd1_z_depart d
),
br_one as (
    select b.id, b.c_shortlabel as filial_rf
    from ods.scd1_z_branch b
),
corp_one as (
    select z.id, z.c_register_gos_reg_num_rec as ogrn
    from ods.scd1_z_cl_corp z
),
tariff_one as (
    select t.id, t.c_name as tariff_name
    from ods.scd1_z_r2_tariff_plan t
),
retl_mcc as (
    select b.n_agr,
           count(distinct cast(m.c_nmrc as string)) as retl_cnt,
           min(cast(m.n_mcc as string)) as mcc_any,
           count(distinct cast(m.n_mcc as string)) as mcc_variants_cnt
    from base b
    left join ods_alpha.scd1_merchants m
      on m.n_cmp = b.n_cmp_client
     and m.c_nmrc is not null
    group by b.n_agr
),
terminals as (
    select b.n_agr,
           count(distinct cast(t.c_nter as string)) as term_cnt
    from base b
    left join ods_alpha.scd1_merchants m
      on m.n_cmp = b.n_cmp_client
     and m.c_nmrc is not null
    left join ods_alpha.scd1_pos_terminals t
      on t.c_nmrc = m.c_nmrc
     and t.c_nter is not null
    group by b.n_agr
),
amort as (
    select b.n_agr,
           sum(coalesce(cast(am.amortization_monthly as double), 0.0)) as amortization
    from base b
    left join ods_alpha.scd1_merchants m
      on m.n_cmp = b.n_cmp_client
     and m.c_nmrc is not null
    left join ods_alpha.scd1_pos_terminals t
      on t.c_nmrc = m.c_nmrc
     and t.c_nter is not null
    left join sandbox_ai.shestopalov_terminal_amortization_oneoff am
      on cast(am.c_nter as string) = cast(t.c_nter as string)
    group by b.n_agr
),
trx_base as (
    select t.n_trx, cast(t.n_amt_src as double) as n_amt_src
    from ods_alpha.scd1_trx t
    where to_date(cast(t.d_trx_orig as timestamp)) between cast('{month_start}' as date) and cast('{month_end}' as date)
      and t.c_nter is not null
      and coalesce(t.ods_deleted_flg, '0') <> '1'
),
trx_join as (
    select cast(ta.n_agr as string) as n_agr,
           tb.n_trx,
           tb.n_amt_src,
           coalesce(cast(ta.n_amt_tax as double), 0.0) as n_amt_tax
    from trx_base tb
    join ods_alpha.scd1_trx_acq ta
      on ta.n_trx = tb.n_trx
),
trx_full as (
    select tj.n_agr,
           tj.n_trx,
           tj.n_amt_src,
           tj.n_amt_tax,
           coalesce(cast(ints.n_amt_fee as double), 0.0) as n_amt_fee
    from trx_join tj
    left join ods_alpha.scd1_trx_int ints
      on ints.n_trx = tj.n_trx
),
trx_agg as (
    select n_agr,
           count(distinct n_trx) as trx_cnt,
           sum(n_amt_src) as trx_sum,
           sum(n_amt_tax) as commission_from_ops,
           sum(n_amt_fee) as int_component
    from trx_full
    group by n_agr
),
fix_r2 as (
    select cast(m.id as string) as agr_id_norm,
           max(cast(tf.c_summa as double)) as fix_comiss,
           count(distinct cast(tf.c_summa as string)) as fix_variants_cnt
    from ods.scd1_z_r2_ip_merchants m
    left join ods.scd1_z_r2_tariff_tune tt on tt.c_tariff_plan = m.c_tariff_plan
    left join ods.scd1_z_r2_tariff_fix tf on tt.c_tariff = tf.id
    where m.id is not null
    group by cast(m.id as string)
)
select
    b.inn_norm,
    b.client_name,
    b.contract_norm,
    b.contract_number,
    b.agr_id_norm,
    b.n_agr,
    b.d_valid_from,
    b.d_valid_to,
    co.ogrn,
    br.filial_rf,
    dep.vsp_name,
    dep.vsp_code,
    trf.tariff_name,
    rm.mcc_any,
    coalesce(rm.mcc_variants_cnt, 0) as mcc_variants_cnt,
    coalesce(rm.retl_cnt, 0) as retl_cnt,
    coalesce(te.term_cnt, 0) as term_cnt,
    coalesce(tx.trx_cnt, 0) as trx_cnt,
    coalesce(tx.trx_sum, 0.0) as trx_sum,
    cast(coalesce(te.term_cnt, 0) as double) * {aur_rate} as aur,
    coalesce(am.amortization, 0.0) as amortization,
    coalesce(tx.commission_from_ops, 0.0) as commission_from_ops,
    coalesce(fr.fix_comiss, 0.0) as commission_monthly,
    coalesce(fr.fix_variants_cnt, 0) as fix_variants_cnt,
    coalesce(tx.int_component, 0.0) as int_component,
    (coalesce(tx.commission_from_ops, 0.0) + coalesce(fr.fix_comiss, 0.0)) as commission_total,
    (coalesce(tx.commission_from_ops, 0.0) + coalesce(fr.fix_comiss, 0.0) - coalesce(tx.int_component, 0.0)) as nod_te,
    (
      coalesce(tx.commission_from_ops, 0.0) + coalesce(fr.fix_comiss, 0.0) - coalesce(tx.int_component, 0.0)
      - cast(coalesce(te.term_cnt, 0) as double) * {aur_rate}
      - coalesce(am.amortization, 0.0)
    ) as fin_result_te,
    case when coalesce(tx.trx_sum, 0.0) <> 0 then coalesce(tx.commission_from_ops, 0.0) / tx.trx_sum * 100.0 end as avg_acq_pct,
    case when coalesce(tx.trx_sum, 0.0) <> 0 then coalesce(tx.int_component, 0.0) / tx.trx_sum * 100.0 end as avg_irf_pct
from base b
left join r2_one r2 on r2.agr_id_norm = b.agr_id_norm
left join dep_one dep on dep.id = r2.c_depart
left join br_one br on br.id = dep.c_filial
left join corp_one co on co.id = r2.c_cl_org
left join tariff_one trf on trf.id = r2.c_tariff_plan
left join retl_mcc rm on rm.n_agr = b.n_agr
left join terminals te on te.n_agr = b.n_agr
left join trx_agg tx on tx.n_agr = b.n_agr
left join fix_r2 fr on fr.agr_id_norm = b.agr_id_norm
left join amort am on am.n_agr = b.n_agr
"""

with imp:
    attrs_df = imp.fetch(sql_attrs)

attrs_df['inn_norm'] = attrs_df['inn_norm'].apply(normalize_inn)
attrs_df['contract_norm'] = attrs_df['contract_norm'].apply(normalize_contract)
attrs_df['agr_id_norm'] = attrs_df['agr_id_norm'].apply(normalize_agr_id)

num_cols = [
    'retl_cnt', 'term_cnt', 'trx_cnt', 'trx_sum', 'aur', 'amortization',
    'commission_from_ops', 'commission_monthly', 'commission_total', 'int_component',
    'nod_te', 'fin_result_te', 'avg_acq_pct', 'avg_irf_pct', 'fix_variants_cnt', 'mcc_variants_cnt'
]
for c in num_cols:
    if c in attrs_df.columns:
        attrs_df[c] = pd.to_numeric(attrs_df[c], errors='coerce')

display(pd.DataFrame([
    {'metric': 'rows_total', 'value': len(attrs_df)},
    {'metric': 'unique_inn_contract', 'value': attrs_df[['inn_norm', 'contract_norm']].drop_duplicates().shape[0]},
    {'metric': 'rows_without_agr_id', 'value': int(attrs_df['agr_id_norm'].isna().sum()) if 'agr_id_norm' in attrs_df.columns else None},
]))
display(attrs_df.head(30))
